# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata
print(metadata)
print(f"\nName: {metadata.name if hasattr(metadata, 'name') else ''}")
print(f"Description: {metadata.description if hasattr(metadata, 'description') else ''}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# List all record sets and their field IDs

record_sets = dataset.metadata.record_set

print("Available Record Sets:")
for rs in record_sets:
    print(f"- RecordSet name: {getattr(rs, 'name', '')}    @id: {getattr(rs, '@id', '')}")
    print(f"  Description: {getattr(rs, 'description', '')}")
    field_ids = []
    for f in getattr(rs, 'field', []):
        fid = getattr(f, '@id', '')
        fname = getattr(f, 'name', '')
        fdesc = getattr(f, 'description', '')
        field_ids.append((fid, fname, fdesc))
    print("  Fields (by @id):")
    for f in field_ids:
        print(f"    @id: {f[0]}, name: {f[1]}, description: {f[2]}")
    print("")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Replace these with valid record set @ids from the overview step
record_set_ids = [rs['@id'] if isinstance(rs, dict) else getattr(rs, '@id', None) for rs in dataset.metadata.record_set]
dataframes = {}

for record_set_id in record_set_ids:
    print(f"Loading records for record set: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded dataframe with shape {df.shape}.")
        print(f"Columns for {record_set_id}:", df.columns.tolist())
        display(df.head())
    else:
        print(f"No records found for {record_set_id}")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section should include operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Example: Choose a numeric field from one of the DataFrames

# We'll loop to find the first available DataFrame with numeric columns
chosen_record_set_id = None
numeric_field = None
for rs_id, df in dataframes.items():
    numeric_cols = df.select_dtypes(include=np.number).columns.tolist()
    if numeric_cols:
        chosen_record_set_id = rs_id
        numeric_field = numeric_cols[0]
        break

if chosen_record_set_id and numeric_field:
    print(f"Using record set: {chosen_record_set_id}")
    print(f"Using numeric field for analysis: {numeric_field}")
    threshold = np.percentile(dataframes[chosen_record_set_id][numeric_field].dropna(), 50)
    filtered_df = dataframes[chosen_record_set_id][dataframes[chosen_record_set_id][numeric_field] > threshold].copy()
    print(f"Filtered records with {numeric_field} > {threshold:.2f}:")
    display(filtered_df[[numeric_field]].head())

    # Normalize numeric field
    filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"Normalized {numeric_field} for filtered records:")
    display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Try grouping by a likely categorical column if exists
    categorical_cols = filtered_df.select_dtypes(include='object').columns.tolist()
    group_field = None
    for col in categorical_cols:
        if filtered_df[col].nunique() < 20 and col != numeric_field:
            group_field = col
            break

    if group_field:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
        print(f"Grouped data by {group_field} (mean {numeric_field}):")
        display(grouped_df)
else:
    print("No suitable DataFrame with a numeric field found to perform EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if chosen_record_set_id and numeric_field:
    df = dataframes[chosen_record_set_id]
    plt.figure(figsize=(9,5))
    sns.histplot(df[numeric_field].dropna(), bins=40, kde=True)
    plt.title(f"Distribution of {numeric_field} in record set {chosen_record_set_id}")
    plt.xlabel(numeric_field)
    plt.ylabel("Frequency")
    plt.show()

    if group_field:
        plt.figure(figsize=(12,6))
        sns.boxplot(x=group_field, y=numeric_field, data=df)
        plt.title(f"{numeric_field} by {group_field} in record set {chosen_record_set_id}")
        plt.xticks(rotation=45)
        plt.show()
else:
    print("No visualization generated (no suitable data found).")

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

* In this notebook, we loaded and explored the FAIR² dataset "Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells" using the `mlcroissant` library, accessing metadata and available record sets/fields by their `@id` as specified in the Croissant schema.
* We demonstrated how to extract tabular data into pandas DataFrames, selected a numeric field for EDA, and applied typical filtering and normalization steps.
* Data distributions and relationships (such as group means and visualizations) were investigated where possible based on the dataset's available fields.
* For deeper analysis, refer to the Croissant schema for field meanings and consult domain metadata for example workflow customization.